<a href="https://colab.research.google.com/github/TomerRippin/Machine-Learning/blob/master/Deep_Learning_Custom_DataLoader_Dataset_Training_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Maman 13 - Deep Learning
## Tomer Rippin - 322230608
---

In [ ]:
from google.colab.drive import mount
mount('/MyDrive')

Mounted at /MyDrive


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

Data Loading and Class Column Creation (Deciles)

In this section, we load the Diabetes dataset and categorize the 'Y' values into 10 deciles stored in a new 'Class' column.

In [ ]:
# Load the dataset
data_path = '/MyDrive/MyDrive/Colab Notebooks/diabetes (1).csv'
df = pd.read_csv(data_path, delimiter='\t')

# Create Target Columns
df['Class'] = pd.qcut(df['Y'], q=10, labels=False)
df['Percentile'] = pd.qcut(df['Y'], q=100, labels=False, duplicates='drop')

# Split into Train and Test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Data loaded. Train size: {len(train_df)}, Test size: {len(test_df)}")
display(df.head())

Data loaded. Train size: 353, Test size: 89


,AGE,SEX,BMI,BP,S1,S2,S3,S4,S5,S6,Y,Class,Percentile
0,59,2,32.1,101.0,157,93.2,38.0,4.0,4.8598,87,151,5,54
1,48,1,21.6,87.0,183,103.2,70.0,3.0,3.8918,69,75,1,19
2,72,2,30.5,93.0,156,93.6,41.0,4.0,4.6728,85,141,5,50
3,24,1,25.3,84.0,198,131.4,40.0,5.0,4.8903,89,206,7,73
4,50,1,23.0,101.0,192,125.4,52.0,4.0,4.2905,80,135,4,47


In [ ]:
class CustomDiabetesDataset(Dataset):
    def __init__(self, dataframe, target_col="Class", exclude_y=False, is_regression=False):
        self.data = dataframe.copy()
        self.target_col = target_col
        self.is_regression = is_regression

        cols_to_exclude = ['Class', 'Percentile']
        if exclude_y and target_col != 'Y':
            cols_to_exclude.append('Y')

        self.feature_cols = [c for c in self.data.columns if c not in cols_to_exclude and c != target_col]

        self.features = torch.tensor(self.data[self.feature_cols].values, dtype=torch.float32)

        if is_regression:
            self.targets = torch.tensor(self.data[target_col].values, dtype=torch.float32).view(-1, 1)
        else:
            self.targets = torch.tensor(self.data[target_col].values, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

In [ ]:
def run_experiment(model, train_loader, test_loader, criterion, optimizer, epochs=50, is_reg=False):
    """Standardized function to train and evaluate models."""
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total, mse_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in test_loader:
            outputs = model(x)
            if is_reg:
                mse_sum += criterion(outputs, y).item() * x.size(0)
            else:
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == y).sum().item()
            total += y.size(0)

    if is_reg:
        return np.sqrt(mse_sum / total) # Returns RMSE
    else:
        return 100 * correct / total # Returns Accuracy %

Running the experiments.

Now we will train 2 nn:
1. With 'Y' in the training data
2. Without 'Y' in the training data

We expect to see a much better result for the first nn as there is a linear depndency between 'Y' and our objective function and target 'Class'.

NN1 - Decile with 'Data leakage' ('Y' in the training data)

In [ ]:
exp1_dataset = CustomDiabetesDataset(train_df)
exp1_test_dataset = CustomDiabetesDataset(test_df)

exp1_train_loader = DataLoader(exp1_dataset, batch_size=10, shuffle=True)
exp1_test_loader = DataLoader(exp1_test_dataset, batch_size=10, shuffle=False)

# Print a minibatch
print(next(iter(exp1_train_loader)))

[tensor([[ 44.0000,   2.0000,  38.2000, 123.0000, 201.0000, 126.6000,  44.0000,
           5.0000,   5.0239,  92.0000, 308.0000],
        [ 66.0000,   2.0000,  26.2000, 114.0000, 255.0000, 185.0000,  56.0000,
           4.5500,   4.2485,  92.0000,  63.0000],
        [ 59.0000,   2.0000,  23.6000,  73.0000, 180.0000, 107.4000,  51.0000,
           4.0000,   4.6821,  84.0000,  88.0000],
        [ 47.0000,   1.0000,  28.6000,  97.0000, 164.0000,  90.6000,  56.0000,
           3.0000,   4.4659,  88.0000, 121.0000],
        [ 28.0000,   1.0000,  24.2000,  93.0000, 174.0000, 106.4000,  54.0000,
           3.0000,   4.2195,  84.0000, 144.0000],
        [ 58.0000,   2.0000,  25.5000, 112.0000, 163.0000, 110.6000,  29.0000,
           6.0000,   4.7622,  86.0000, 257.0000],
        [ 29.0000,   1.0000,  35.0000,  98.3300, 204.0000, 142.6000,  50.0000,
           4.0800,   4.0431,  91.0000, 200.0000],
        [ 43.0000,   1.0000,  26.8000, 123.0000, 193.0000, 102.2000,  67.0000,
           3.0000

Parameter selection:
I chose 2 hidden layers with sizes 24, 12 as my hidden layers as they provded my with consistantly good results.

I used SGD as my optimizer with 0.01 learning rate parameter as it the optimization we learned in the book.

I ran 100 epochs as the Loss didnt improve after that in a drastic way.

In [ ]:
exp1_model = nn.Sequential(
    nn.Linear(len(exp1_dataset.feature_cols), 24),
    nn.ReLU(),
    nn.Linear(24, 12),
    nn.ReLU(),
    nn.Linear(12, 10),
    nn.Softmax(dim=1)
)

exp1_criterion = nn.CrossEntropyLoss()
exp1_optimizer = optim.SGD(exp1_model.parameters(), lr=0.01)

In [ ]:
accuracy = run_experiment(exp1_model, exp1_train_loader, exp1_test_loader,
                          exp1_criterion, exp1_optimizer, epochs=100)

accuracy

23.59550561797753

NN2 - Decile without 'Y' in the training

In [ ]:
exp2_dataset = CustomDiabetesDataset(train_df, exclude_y=True)
exp2_test_dataset = CustomDiabetesDataset(test_df, exclude_y=True)

exp2_train_loader = DataLoader(exp2_dataset, batch_size=10, shuffle=True)
exp2_test_loader = DataLoader(exp2_test_dataset, batch_size=10, shuffle=False)

NN2 - Parameter selection:

I chose the network to be similar to the exp1 netowrk to better compare the differences between the networks.

In [ ]:
exp2_model = nn.Sequential(
    nn.Linear(len(exp2_dataset.feature_cols), 24),
    nn.ReLU(),
    nn.Linear(24, 12),
    nn.ReLU(),
    nn.Linear(12, 10),
    nn.Softmax(dim=1)
)

exp2_criterion = nn.CrossEntropyLoss()
exp2_optimizer = optim.SGD(exp2_model.parameters(), lr=0.01)

In [ ]:
accuracy = run_experiment(exp2_model, exp2_train_loader, exp2_test_loader,
                          exp2_criterion, exp2_optimizer, epochs=100)

accuracy

14.606741573033707

We can clearly see that we got much better results in the NN that learned on the data with 'Y' in it.

This is called 'Data Leakage':
Data Leakage occurs when information from outside the training dataset—specifically from the test set or future data—is inadvertently used to train a model. This makes the model appear highly accurate during development, but causes it to fail or perform poorly when deployed on real-world, unseen data.

In our example this is the 'Y' column which is equivelent to the 'Class' column.

##### Part 2 #####

In this part we will try and do it with percentiles instead of deciles.

NN3 - percentile instead of decile with 'Data leakage' ('Y' in training data')

In [ ]:
exp3_dataset = CustomDiabetesDataset(train_df, target_col='Percentile')
exp3_test_dataset = CustomDiabetesDataset(test_df, target_col='Percentile')

exp3_train_loader = DataLoader(exp3_dataset, batch_size=10, shuffle=True)
exp3_test_loader = DataLoader(exp3_test_dataset, batch_size=10, shuffle=False)

NN3 - Paramater selection:
I didnt have much sucess in the start by using a similar network layout as the previues experiments.

after some tries i succefuly found this layout with 5% accuarcy and much better loss than previeus network models.
I chose a smaller learning rate and higher epochs as i saw that the network had better success in learning over time in smaller steps.

In [ ]:
exp3_model = nn.Sequential(
    nn.Linear(len(exp3_dataset.feature_cols), 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 100),
    nn.Softmax(dim=1)
)

exp3_criterion = nn.CrossEntropyLoss()
exp3_optimizer = optim.SGD(exp3_model.parameters(), lr=0.001)

In [ ]:
accuracy = run_experiment(exp3_model, exp3_train_loader, exp3_test_loader,
                          exp3_criterion, exp3_optimizer, epochs=500)

accuracy

5.617977528089888

NN4 - percentile instead of decile without 'Data leakage'

In [ ]:
exp4_dataset = CustomDiabetesDataset(train_df, target_col='Percentile', exclude_y=True)
exp4_test_dataset = CustomDiabetesDataset(test_df, target_col='Percentile', exclude_y=True)

exp4_train_loader = DataLoader(exp4_dataset, batch_size=10, shuffle=True)
exp4_test_loader = DataLoader(exp4_test_dataset, batch_size=10, shuffle=False)

NN4 - Parameter selection.
For better comparison in this experiment too i decided to have similar network layer as exp3.

In [ ]:
# Re-defining Exp 4 with Adam and a better LR
exp4_model = nn.Sequential(
    nn.Linear(len(exp4_dataset.feature_cols), 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 100),
    nn.Softmax(dim=1)
)

exp4_criterion = nn.CrossEntropyLoss()
exp4_optimizer = optim.SGD(exp4_model.parameters(), lr=0.001)

In [ ]:
accuracy = run_experiment(exp4_model, exp4_train_loader, exp4_test_loader,
                          exp4_criterion, exp4_optimizer, epochs=500)

accuracy

3.3707865168539324

Comparison:

We can see that the acuuarcy in NN3 is 5.6% as in NN4 is 3.4%. this is a huge gap and it is conssistant with the 'Data Leakage'.

We can also see that the accuarcy result in both NN3 and NN4 are much lower than NN1 and NN2. we can explain that phenomena due to the fact that classifieng to 100 classes is much harder than to 10 classes as you need to be much more precise with your prediction.

In our models when we have such little data to train (~350 samples) on it is really difficult to catagorize it for 100 classes.

Part 3: Regression model.

In [ ]:
def run_regression_experiment(target_col, exclude_y):
    train_ds = CustomDiabetesDataset(train_df, target_col=target_col, exclude_y=exclude_y, is_regression=True)
    test_ds = CustomDiabetesDataset(test_df, target_col=target_col, exclude_y=exclude_y, is_regression=True)

    train_loader = DataLoader(train_ds, batch_size=10, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=10, shuffle=False)

    model = nn.Sequential(
        nn.Linear(len(train_ds.feature_cols), 24),
        nn.ReLU(),
        nn.Linear(24, 12),
        nn.ReLU(),
        nn.Linear(12, 1)
    )

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    rmse = run_experiment(model, train_loader, test_loader, criterion, optimizer, epochs=100, is_reg=True)
    return rmse

rmse_with_y = run_regression_experiment('Percentile', exclude_y=False)
rmse_no_y = run_regression_experiment('Percentile', exclude_y=True)

print(f"Regression RMSE (With Y): {rmse_with_y:.4f}")
print(f"Regression RMSE (No Y): {rmse_no_y:.4f}")
print(f"Leakage Factor: {rmse_no_y / rmse_with_y:.2f}x worse without Y")

Regression RMSE (With Y): 2.0627
Regression RMSE (No Y): 22.9255
Leakage Factor: 11.11x worse without Y


I chose to run the network on pretty simmilar structure to the networks before but i chose the Adam optimizer as it worked much better.

We can clearly see that the RMSE with 'Y' is much smaller than witout 'Y' which is understandable as we want to predict 'Y' we cannot use it in our training, its "Cheating!".

### Conclusion and Discussion ###
In this assigment we learned how to buled custom datasets and dataloaders.

We also build our first Deep Network using PyTorch to classify diabetes risk.

We build several diferent networks to learn about 'Data Leakage' - the concept that you learn on the data that you want to classify which gives you a massive advantage and better accuarcy and results but it is wrong.

We also learned about the fact that it is easier to classify to less classes, especcialy when you have small dataset to learn on. when you classify to more classes the model will be less confident in the results and have smaller accuarcy.

We also tried to build a regression network and saw in a different way the difference between learning with 'Data Leakage' and without.